# 03 - MACE Training
Train the message-passing MACE model with the same training setup as ACE and model-specific architecture choices.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.mace_wrapper import MACEWrapper
from src.trainer import BenchmarkTrainer

In [2]:
# Load Data
train_ds = MDTrajectoryDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MDTrajectoryDataset("../data/val.extxyz", cutoff=5.0)
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize MACE model (shared training setup, model-specific architecture)
model = MACEWrapper(
    num_elements=120,
    r_max=5.0,
    num_radial=8,
    l_max=2,
    num_blocks=2,  # 2 layers of message passing
    node_dim=16
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)
print(f"Training on device: {trainer.device}\n")

Training on device: cuda



In [4]:
# Train + held-out test evaluation
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/mace_metrics.csv", index=False)

test_metrics = trainer.test_epoch(test_loader)
mace_test_df = pd.DataFrame([test_metrics])
mace_test_df.to_csv("../data/mace_test_metrics.csv", index=False)

# Save the trained model state
torch.save(model.state_dict(), "../data/mace_model.pth")

metrics_df.head()

Epoch 000 | Time: 17.86s | Train E MAE: 488.14 meV/atom | Train F MAE: 181.49 meV/Å | Val E MAE: 281.31 meV/atom | Val F MAE: 48.25 meV/Å
Epoch 001 | Time: 15.92s | Train E MAE: 109.51 meV/atom | Train F MAE: 20.29 meV/Å | Val E MAE: 52.48 meV/atom | Val F MAE: 10.75 meV/Å
Epoch 002 | Time: 15.87s | Train E MAE: 19.27 meV/atom | Train F MAE: 7.10 meV/Å | Val E MAE: 6.65 meV/atom | Val F MAE: 5.08 meV/Å
Epoch 003 | Time: 15.84s | Train E MAE: 3.60 meV/atom | Train F MAE: 4.82 meV/Å | Val E MAE: 1.50 meV/atom | Val F MAE: 4.60 meV/Å
Epoch 004 | Time: 15.77s | Train E MAE: 0.77 meV/atom | Train F MAE: 4.28 meV/Å | Val E MAE: 0.28 meV/atom | Val F MAE: 3.91 meV/Å
Epoch 005 | Time: 16.00s | Train E MAE: 0.13 meV/atom | Train F MAE: 3.96 meV/Å | Val E MAE: 0.15 meV/atom | Val F MAE: 3.71 meV/Å
Epoch 006 | Time: 15.63s | Train E MAE: 0.09 meV/atom | Train F MAE: 3.75 meV/Å | Val E MAE: 0.05 meV/atom | Val F MAE: 3.58 meV/Å
Epoch 007 | Time: 15.63s | Train E MAE: 0.06 meV/atom | Train F MAE: 3

,epoch,loss,e_mae,f_mae,time,val_loss,val_e_mae,val_f_mae
0,0,41.415935,488.142528,181.491907,17.861308,5.493643,281.305428,48.250417
1,1,1.209132,109.505010,20.289382,15.916003,0.194355,52.482167,10.745683
2,2,0.045326,19.270998,7.103585,15.867934,0.007372,6.652803,5.081883
3,3,0.005555,3.604658,4.822040,15.837552,0.003644,1.495639,4.599122
4,4,0.003328,0.769635,4.279994,15.767012,0.003073,0.282233,3.913485


In [5]:
# Check model size (number of parameters)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 417,825
Trainable Parameters: 417,825
